# Day 09: TensorRT-LLM — Compilation & Plugin System
> *Inference Engineering* — Chapter 4.3.3 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 06 (ONNX/TRT), Day 07 (vLLM)


## Concept Overview

TensorRT-LLM is NVIDIA's framework for compiling LLMs to maximally optimized TensorRT engines. Unlike generic TensorRT, it understands LLM-specific patterns: it handles multi-head attention, KV caching, in-flight batching, and tensor parallelism natively. The plugin system provides hand-tuned CUDA kernels for operations TensorRT's general optimizer can't handle (e.g., FlashAttention, rotary embeddings, sampling). The compilation produces a GPU-architecture-specific engine that fuses operations and selects the fastest kernel for each op at each shape.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. TensorRT-LLM Architecture

The build flow: Python model definition → C++/CUDA network → TensorRT engine. Key phases: (1) graph construction, (2) optimization passes, (3) engine serialization. Plugins inject hand-written CUDA into the TRT graph at specific nodes.


In [ ]:
# TRT-LLM build pipeline (conceptual — shows structure without requiring install)
pipeline_stages = [
    ('Model Definition',     'Python API: define layers, dtypes, TP/PP config'),
    ('Network Construction', 'C++ NetworkDefinition + plugin registration'),
    ('Optimization Passes',  'Layer fusion, precision calibration, tactic selection'),
    ('Engine Compilation',   'CUDA kernel selection per op per shape'),
    ('Serialization',        'Write .engine file (GPU-arch specific)'),
    ('Runtime',              'TensorRT runtime + in-flight batching loop'),
]
print('TensorRT-LLM Build Pipeline:')
print('=' * 70)
for i, (stage, desc) in enumerate(pipeline_stages):
    print(f'  Step {i+1}: {stage:<30} | {desc}')

print()
print('Key TRT-LLM plugins:')
plugins = [
    ('gpt_attention', 'Fused MHA/MQA/GQA with KV cache management'),
    ('rmsnorm_quantization', 'Fused RMSNorm + FP8 quantization'),
    ('smooth_quant_gemm', 'INT8 GEMM with SmoothQuant scaling'),
    ('weight_only_quant_matmul', 'INT4/INT8 weight-only GEMM'),
    ('lookup', 'Token embedding lookup'),
    ('lora', 'Low-rank adaptation weights'),
]
for name, desc in plugins:
    print(f'  {name:<35} {desc}')


## 2. Compilation Speedup Sources

TRT-LLM gains come from multiple orthogonal sources: kernel selection, fusion, and precision.


In [ ]:
# Model the speedup from each optimization independently
optimizations = {
    'Baseline (eager PyTorch FP32)': 1.0,
    '+ FP16 precision':              1.8,
    '+ FlashAttention plugin':        2.3,
    '+ Kernel fusion (FFN)':          2.8,
    '+ INT8 SmoothQuant':             4.2,
    '+ INT4 weight-only':             5.1,
    '+ In-flight batching':           8.0,
    '+ TP=2 tensor parallel':        14.0,
}

names = list(optimizations.keys())
speedups = list(optimizations.values())

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(names)), speedups, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(names))))
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Throughput Speedup vs Baseline')
ax.set_title('TensorRT-LLM Cumulative Optimizations')
for bar, val in zip(bars, speedups):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.1f}x', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('trt_llm_speedups.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved trt_llm_speedups.png')


## 3. Tactic Selection

TensorRT benchmarks multiple kernel implementations (tactics) for each operation and selects the fastest for the target GPU. This is done at engine build time with real measurements. The profile file caches tactic selections to avoid repeated profiling.


In [ ]:
# Simulate tactic selection for a GEMM op
import random
random.seed(42)

def simulate_tactic_selection(m, k, n, dtype='fp16'):
    """Simulate timing multiple GEMM tactics."""
    # Different kernel implementations have different strengths
    tactics = {
        'cutlass_sm90_gemm_fp16':     random.uniform(0.8, 1.2),
        'cublas_gemm_algo_0':         random.uniform(1.0, 1.5),
        'cublas_gemm_algo_5':         random.uniform(0.9, 1.3),
        'cutlass_splitk_2':           random.uniform(0.7, 1.1),
        'cutlass_warpspecialized':    random.uniform(0.6, 1.0),
        'cublas_gemm_algo_default':   random.uniform(1.1, 1.6),
    }
    print(f'  Tactic profiling for GEMM [{m}x{k}] x [{k}x{n}] ({dtype}):')
    results = {}
    for tactic, latency_ms in tactics.items():
        results[tactic] = latency_ms
        print(f'    {tactic:<40} {latency_ms:.3f} ms')
    best = min(results, key=results.get)
    print(f'  Selected: {best} ({results[best]:.3f} ms)')
    return best

print('TensorRT tactic selection simulation:')
print()
for (m, k, n) in [(4096, 4096, 4096), (1, 4096, 14336), (32, 4096, 14336)]:
    simulate_tactic_selection(m, k, n)
    print()


## Experiments: Try These

1. **Profile PyTorch vs TRT**: Use `torch.utils.benchmark` to compare PyTorch eager attention vs `F.scaled_dot_product_attention` (which uses FlashAttention). This is the core of what TRT-LLM's gpt_attention plugin does.
2. **Precision sweep**: Benchmark FP32 vs FP16 vs BF16 matrix multiplies across sizes. When does precision change affect speed vs accuracy?
3. **Engine size vs accuracy**: Research TRT-LLM's INT4 AWQ support. What is the perplexity delta between FP16 and INT4 for Llama-3-8B?


## Key Takeaways

- TRT-LLM combines offline compilation, tactic selection, and LLM-specific plugins to produce maximally optimized inference engines.
- The plugin system injects hand-tuned CUDA for operations general TensorRT can't optimize (FlashAttention, RoPE, sampling).
- Tactic selection is a compile-time benchmark: TRT profiles multiple CUDA kernel implementations and picks the fastest for each shape/dtype/GPU combination.
- Cumulative optimizations (FP16 + fusion + quantization + TP) yield 10-14x throughput vs eager PyTorch FP32 for production workloads.

## References
- *Inference Engineering* Ch 4.3.3 — Philip Kiely, Baseten Books 2026
- NVIDIA TensorRT-LLM GitHub repository
- NVIDIA GTC 2024: TensorRT-LLM Deep Dive
